# Hospital Laboratory Dataset: Regex and Text Preprocessing

**Project objective:** Explore a hospital laboratory dataset, perform basic cleaning, use regular expressions (Regex) to extract information from laboratory report text, visualize the results, and export the processed data.

**Scope:** Python, Pandas, Regex, and Matplotlib. This project does not include machine learning or predictive modelling.

**Input file:** `synthetic_hospital_lab_dataset_1000.csv`

> Keep the original dataset unchanged. The notebook creates a working copy and saves results in an `output` folder.


## Setup: Import libraries and configure paths

**Objective:** Import the required libraries and define the input and output locations.

**Input:** The original CSV file.

**Method:** Use Pandas for data handling, `re` for regular expressions, and Matplotlib for charts.

**Expected output:** Libraries imported and output folder created.

# Phase 1 — Reading and understanding the dataset

## Step 1.1: Load the dataset

**Objective:** Read the CSV file into a Pandas DataFrame.

**Input:** `synthetic_hospital_lab_dataset_1000.csv`

**Method:** Use `pd.read_csv()`.

**Expected output:** The dataset is loaded and its first five rows are displayed.

In [ ]:
df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully.")
display(df.head())

## Step 1.2: Inspect dataset structure

**Objective:** Understand the number of records, columns, column names, and data types.

**Input:** Loaded DataFrame.

**Method:** Use `shape`, `columns`, and `info()`.

**Expected output:** Dataset dimensions and data type information.

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

## Step 1.3: Check missing values and duplicates

**Objective:** Identify incomplete records and duplicate rows.

**Input:** Loaded DataFrame.

**Method:** Use `isna().sum()` and `duplicated().sum()`.

**Expected output:** Missing-value counts for each column and the number of duplicate rows. Do not fill missing laboratory measurements with invented values.

In [ ]:
missing_summary = df.isna().sum().rename("missing_count").to_frame()
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df) * 100).round(2)

print("Missing values:")
display(missing_summary)

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

## Step 1.4: Explore categories and report text

**Objective:** Understand which laboratory tests appear and inspect the unstructured report descriptions.

**Input:** `Test_Name` and `Lab_Report_Text`.

**Method:** Use value counts and display example report strings.

**Expected output:** A list of common tests and examples of inconsistent report formatting.

In [ ]:
print("Laboratory test counts:")
display(df["Test_Name"].value_counts(dropna=False))

print("\nExample laboratory report text:")
for report in df["Lab_Report_Text"].dropna().head(10):
    print("-", report)

## Phase 1 summary

At this point, we have loaded the original dataset and examined its structure, missing values, duplicates, test categories, and raw report text.

**Phase 1 output:** A basic understanding of the dataset and its initial data quality.

In [ ]:
phase1_summary = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Duplicate rows", "Total missing cells"],
    "Value": [df.shape[0], df.shape[1], int(df.duplicated().sum()), int(df.isna().sum().sum())]
})
display(phase1_summary)

# Phase 2 — Data cleaning and regular expressions

## Step 2.1: Create a working copy

**Objective:** Protect the original DataFrame and prepare a separate copy for processing.

**Input:** Original DataFrame `df`.

**Method:** Use `.copy()`.

**Expected output:** A working DataFrame called `clean_df`.

In [ ]:
clean_df = df.copy()
print("Working copy created:", clean_df.shape)

## Step 2.2: Basic text cleaning

**Objective:** Make report text more consistent without destroying meaningful characters such as decimal points, slashes, percentage signs, or unit symbols.

**Input:** `Lab_Report_Text`.

**Method:** Convert text to lowercase, trim leading/trailing spaces, and collapse repeated whitespace. Missing text remains missing.

**Expected output:** A new column named `Lab_Report_Text_Clean`.

In [ ]:
def basic_clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_df["Lab_Report_Text_Clean"] = clean_df["Lab_Report_Text"].apply(basic_clean_text)

display(clean_df[["Lab_Report_Text", "Lab_Report_Text_Clean"]].head(10))

## Step 2.3: Understand basic Regex patterns

**Objective:** Practise common Regex symbols before extracting information.

| Pattern | Meaning |
|---|---|
| `\d` | A digit from 0 to 9 |
| `\d+` | One or more digits |
| `\s` | A whitespace character |
| `\s*` | Zero or more whitespace characters |
| `[A-Za-z]+` | One or more letters |
| `\.` | A literal decimal point |
| `(...)` | Capture a matching group |

**Expected output:** Test patterns against a sample report.

In [ ]:
sample_text = "Test=HbA1c | Result: 6.3 % | collected 01-26-2023"

print("Original:", sample_text)
print("Digits:", re.findall(r"\d+", sample_text))
print("Words:", re.findall(r"[A-Za-z]+", sample_text))
print("Decimal result:", re.findall(r"\d+(?:\.\d+)?", sample_text))

## Step 2.4: Extract test name, result, unit, and collection date

**Objective:** Extract structured information from the unstructured laboratory report.

**Input:** `Lab_Report_Text_Clean`.

**Method:** Use `re.search()` with capture groups. The patterns allow different separators and common date layouts.

**Important:** The date is extracted as text only. Dates such as `05/03/2024` can be ambiguous (day-first or month-first), so this notebook does not guess their meaning.

**Expected output:** Four new columns: `Extracted_Test_Name`, `Extracted_Result`, `Extracted_Unit`, and `Extracted_Date_Text`.

In [1]:
# Test name: stop when the Result field begins, allowing |, ;, or , separators.
TEST_PATTERN = re.compile(
    r"Test\s*=\s*(.*?)\s*(?:[|;,]\s*)?Result\s*:",
    flags=re.IGNORECASE
)

# Numeric result and optional unit (e.g. 6.3 %, 139.45 mmol/L, 5.66 mg/L, 10^9/L).
RESULT_PATTERN = re.compile(
    r"Result\s*:\s*([-+]?\d+(?:\.\d+)?)\s*([A-Za-z0-9^/%]+)?",
    flags=re.IGNORECASE
)

# Capture date text in YYYY-MM-DD / YYYY/MM/DD or day/month/year-style formats.
DATE_PATTERN = re.compile(
    r"collected\s+(\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{4})",
    flags=re.IGNORECASE
)

def extract_fields(report):
    if pd.isna(report):
        return pd.Series({
            "Extracted_Test_Name": pd.NA,
            "Extracted_Result": pd.NA,
            "Extracted_Unit": pd.NA,
            "Extracted_Date_Text": pd.NA
        })

    text = str(report)

    test_match = TEST_PATTERN.search(text)
    result_match = RESULT_PATTERN.search(text)
    date_match = DATE_PATTERN.search(text)

    test_name = test_match.group(1).strip(" |;,\t") if test_match else pd.NA
    result = float(result_match.group(1)) if result_match else pd.NA
    unit = result_match.group(2) if result_match and result_match.group(2) else pd.NA
    date_text = date_match.group(1) if date_match else pd.NA

    return pd.Series({
        "Extracted_Test_Name": test_name,
        "Extracted_Result": result,
        "Extracted_Unit": unit,
        "Extracted_Date_Text": date_text
    })

extracted = clean_df["Lab_Report_Text_Clean"].apply(extract_fields)
clean_df = pd.concat([clean_df, extracted], axis=1)

display(clean_df[[
    "Lab_Report_Text_Clean",
    "Extracted_Test_Name",
    "Extracted_Result",
    "Extracted_Unit",
    "Extracted_Date_Text"
]].head(15))

NameError: name 're' is not defined

## Step 2.5: Evaluate extraction results

**Objective:** Measure how often Regex found each field.

**Input:** Newly extracted columns.

**Method:** Count non-missing extracted values and calculate the percentage of report rows with a match.

**Expected output:** An extraction summary. A missing extraction is a signal to inspect the source text or improve the pattern; it is not automatically proof that the original record is wrong.

In [ ]:
extracted_columns = [
    "Extracted_Test_Name",
    "Extracted_Result",
    "Extracted_Unit",
    "Extracted_Date_Text"
]

extraction_summary = pd.DataFrame({
    "field": extracted_columns,
    "matched_records": [int(clean_df[col].notna().sum()) for col in extracted_columns]
})
extraction_summary["not_matched"] = len(clean_df) - extraction_summary["matched_records"]
extraction_summary["match_percent"] = (
    extraction_summary["matched_records"] / len(clean_df) * 100
).round(2)

display(extraction_summary)

## Step 2.6: Compare extracted values with existing structured columns

**Objective:** Check whether Regex-extracted test names and numerical results agree with the existing columns.

**Input:** Original structured columns and extracted columns.

**Method:** Compare values where both are available. This is a simple consistency check, not a clinical validation.

**Expected output:** Agreement counts and a sample of records that differ.

In [ ]:
# Compare test names without case sensitivity and surrounding whitespace.
test_both = clean_df["Test_Name"].notna() & clean_df["Extracted_Test_Name"].notna()
test_match = (
    clean_df.loc[test_both, "Test_Name"].astype(str).str.strip().str.casefold()
    == clean_df.loc[test_both, "Extracted_Test_Name"].astype(str).str.strip().str.casefold()
)

# Compare numerical results using a small tolerance for floating-point values.
value_both = clean_df["Test_Value"].notna() & clean_df["Extracted_Result"].notna()
value_match = (
    (clean_df.loc[value_both, "Test_Value"] - clean_df.loc[value_both, "Extracted_Result"]).abs()
    < 1e-8
)

print(f"Test-name agreement: {int(test_match.sum())} / {len(test_match)} comparable records")
print(f"Result agreement: {int(value_match.sum())} / {len(value_match)} comparable records")

display(clean_df.loc[
    test_both & (clean_df["Test_Name"].astype(str).str.strip().str.casefold()
                 != clean_df["Extracted_Test_Name"].astype(str).str.strip().str.casefold()),
    ["Test_Name", "Extracted_Test_Name", "Lab_Report_Text"]
].head(10))

## Phase 2 summary

We created a working copy, standardized report text, and used Regex to extract test names, numerical results, units, and date strings.

**Data handling note:** Missing structured values are preserved. Extracted dates remain text because some date formats are ambiguous.

In [ ]:
print("Processed dataset shape:", clean_df.shape)
print("New extracted columns:", extracted_columns)
display(clean_df[extracted_columns].head())

# Phase 3 — Visualization and saving results

## Step 3.1: Visualize laboratory test distribution

**Objective:** See which laboratory tests occur most frequently.

**Input:** `Test_Name`.

**Method:** Use `value_counts()` and a Matplotlib bar chart.

**Expected output:** A chart showing the number of records for each test.

In [ ]:
test_counts = clean_df["Test_Name"].value_counts(dropna=False)

plt.figure(figsize=(10, 5))
test_counts.plot(kind="bar")
plt.title("Distribution of Laboratory Tests")
plt.xlabel("Test name")
plt.ylabel("Number of records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 3.2: Visualize missing values

**Objective:** Identify which columns contain missing values.

**Input:** All columns in the processed dataset.

**Method:** Count missing values per column and display a bar chart.

**Expected output:** A missing-value chart.

In [ ]:
missing_counts = clean_df.isna().sum()

plt.figure(figsize=(10, 5))
missing_counts.plot(kind="bar")
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Missing value count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

display(missing_counts.rename("missing_count").to_frame())

## Step 3.3: Visualize Regex extraction performance

**Objective:** Compare the number of successful and unsuccessful extractions for each field.

**Input:** `extraction_summary`.

**Method:** Use a stacked bar chart.

**Expected output:** A chart showing matched and unmatched records.

In [ ]:
plot_data = extraction_summary.set_index("field")[["matched_records", "not_matched"]]

plot_data.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Regex Extraction Results")
plt.xlabel("Extracted field")
plt.ylabel("Number of records")
plt.xticks(rotation=30, ha="right")
plt.legend(["Matched", "Not matched"])
plt.tight_layout()
plt.show()

## Step 3.4: Save the cleaned dataset and summaries

**Objective:** Export the processed dataset and project summaries.

**Input:** `clean_df`, `extraction_summary`, and missing-value counts.

**Method:** Save the results as CSV files in the `output` folder.

**Expected output:**
- `hospital_lab_dataset_cleaned.csv`
- `preprocessing_summary.csv`
- `extraction_summary.csv`

In [ ]:
cleaned_file = OUTPUT_DIR / "hospital_lab_dataset_cleaned.csv"
preprocessing_file = OUTPUT_DIR / "preprocessing_summary.csv"
extraction_file = OUTPUT_DIR / "extraction_summary.csv"

clean_df.to_csv(cleaned_file, index=False)

preprocessing_summary = pd.DataFrame({
    "metric": [
        "number_of_rows",
        "number_of_columns_after_processing",
        "duplicate_rows_in_original",
        "total_missing_cells_original"
    ],
    "value": [
        len(clean_df),
        clean_df.shape[1],
        int(df.duplicated().sum()),
        int(df.isna().sum().sum())
    ]
})

preprocessing_summary.to_csv(preprocessing_file, index=False)
extraction_summary.to_csv(extraction_file, index=False)

print("Files saved:")
print(cleaned_file.resolve())
print(preprocessing_file.resolve())
print(extraction_file.resolve())

## Step 3.5: Reload and verify exported file

**Objective:** Confirm that the cleaned CSV was saved and can be read back.

**Input:** `hospital_lab_dataset_cleaned.csv`.

**Method:** Reload it with Pandas and compare its dimensions with the processed DataFrame.

**Expected output:** A successful verification message.

In [ ]:
verified_df = pd.read_csv(cleaned_file)

print("Original processed shape:", clean_df.shape)
print("Reloaded CSV shape:", verified_df.shape)

if verified_df.shape == clean_df.shape:
    print("Verification successful: exported CSV has the expected dimensions.")
else:
    print("Please investigate: dimensions do not match.")

display(verified_df.head())

# Final project conclusion

In this project, we:

1. Read and explored a hospital laboratory dataset.
2. Examined missing values, duplicates, data types, and report text.
3. Applied basic text cleaning.
4. Used Regex to extract test names, numerical results, units, and collection date strings.
5. Evaluated extraction results and compared extracted information with structured columns.
6. Created visualizations and exported the processed data.

## Final deliverables

| Output | Description |
|---|---|
| `hospital_lab_dataset_cleaned.csv` | Original columns plus cleaned text and Regex-extracted fields |
| `preprocessing_summary.csv` | Overall dataset summary |
| `extraction_summary.csv` | Regex matching statistics |
| This notebook | Step-by-step project documentation |

**Important limitation:** Regex extraction is pattern-based. Always inspect unmatched or inconsistent records. Extracted laboratory values and dates are for data-processing practice only and should not be treated as clinical interpretation.